# Limpieza y Normalización de Datos de Permisos de Construcción en San Francisco

## Importar librerías

In [2]:
import pandas as pd
import numpy as np
import re

### Cargar Datos

In [3]:
df = pd.read_csv('../data/permisos_construccion_2.csv', low_memory=False)

### Inspección inicial

In [12]:
n = 15 #  
a = 0 # Parámetros que se fueron cambiando para ver qué columnas tenían valores distintos y hacer análisis
distinct_counts = {col: df[col].nunique(dropna=True) for col in df.columns}

distinct_counts_df = (
    pd.DataFrame(list(distinct_counts.items()), columns=["Variable", "Distinct_Count"])
    .sort_values(by="Distinct_Count", ascending=True)
    .reset_index(drop=True)
)

small_categories = {
    col: df[col].value_counts(dropna=True).to_dict()
    for col, count in distinct_counts.items()
    if  a < count < n
}

small_df = (
    pd.DataFrame(
        [(var, valor, cuenta) 
         for var, subdict in small_categories.items() 
         for valor, cuenta in subdict.items()],
        columns=["Variable", "Value", "Count"]
    )
    .sort_values(by=["Variable", "Count"], ascending=[True, False])
    .reset_index(drop=True)
)

## Análisis de columnas con pocos valores distintos

### Construction Type


Se decidió **unificar y simplificar** la información proveniente de las cuatro columnas iniciales:

- **`Proposed Construction Type Description`**  
- **`Proposed Construction Type`**  
- **`Existing Construction Type Description`**  
- **`Existing Construction Type`**

Tras un proceso de limpieza y normalización, se eliminaron las columnas redundantes de *Description* y se consolidaron las columnas de *Proposed* y *Existing* en una **única columna final** llamada `Construction Type`.  

En esta columna:

- Si existía un valor válido en **Existing**, se conserva con el prefijo **`E`** (ej: `E 3.0`).  
- Si *Existing* estaba vacío pero *Proposed* tenía valor, se conserva con el prefijo **`P`** (ej: `P 1.0`).  
- Si ambas estaban vacías, se deja como `NaN`.  

De esta manera, la información queda unificada, coherente y lista para su uso en el análisis, evitando duplicaciones y garantizando consistencia.

In [5]:
# Convertir a numérico, valores inválidos a NaN
df["Existing Construction Type"] = pd.to_numeric(
    df["Existing Construction Type"], errors="coerce"
)

valid_existing = {1.0, 2.0, 3.0, 4.0, 5.0}
mask_invalid_exist = ~df["Existing Construction Type"].isin(valid_existing)

# Normalizar la Description para comparar de forma robusta
norm_desc = (
    df["Existing Construction Type Description"]
      .astype("string")
      .str.strip()
      .str.lower()
)

# Coincidencia con "wood frame (5)"
mask_wood5 = mask_invalid_exist & norm_desc.str.contains(
    r"^wood\s*frame\s*\(\s*5\s*\)$", regex=True, na=False
)

# Asignar 5.0 cuando la descripción indique wood frame (5)
df.loc[mask_wood5, "Existing Construction Type"] = 5.0

# El resto de los inválidos -> NaN
df.loc[mask_invalid_exist & ~mask_wood5, "Existing Construction Type"] = np.nan



df["Proposed Construction Type"] = pd.to_numeric(
    df["Proposed Construction Type"], errors="coerce"
)

mask_copy = df["Existing Construction Type"].notna()
df.loc[mask_copy, "Proposed Construction Type"] = df.loc[mask_copy, "Existing Construction Type"]


def merge_proposed_existing(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Asegurar que sean numéricos
    df["Proposed Construction Type"] = pd.to_numeric(df["Proposed Construction Type"], errors="coerce")
    df["Existing Construction Type"] = pd.to_numeric(df["Existing Construction Type"], errors="coerce")

    # Crear nueva columna combinada
    df["Construction Type"] = np.where(
        df["Existing Construction Type"].notna(),
        "E " + df["Existing Construction Type"].astype(str),
        np.where(
            df["Proposed Construction Type"].notna(),
            "P " + df["Proposed Construction Type"].astype(str),
            np.nan
        )
    )

    return df

df = merge_proposed_existing(df)

df = df.drop(columns=[
    "Proposed Construction Type Description",
    "Existing Construction Type Description",
    "Proposed Construction Type",
    "Existing Construction Type"
], errors="ignore")


## Permit Type Definition

Corrección de algunos valores que ponía como distintos cuando eran iguales

In [6]:

df["Permit Type Definition"] = (
    df["Permit Type Definition"]
      .astype("string")
      .str.lower()
      .str.strip()                      
      .str.replace(r"\s+", " ", regex=True)  
)

map_dict = {
    "otc alterations permit #": "otc alterations permit",
    "otc alterations permit": "otc alterations permit",
    "additions alterations or repairs": "additions alterations or repairs",
    "new construction wood frame": "new construction wood frame",
    "new construction": "new construction",
    "new construction #": "new construction",
    "sign - erect": "sign - erect",
    "demolitions": "demolitions",
    "wall or painted sign": "wall or painted sign",
    "grade or quarry or fill or excavate": "grade or quarry or fill or excavate"
}

df["Permit Type Definition"] = df["Permit Type Definition"].replace(map_dict)

## Supervisor District

Corrección de formato

In [7]:
df["Supervisor District"] = (
    df["Supervisor District"]
      .astype("string")
      .str.strip()
      .str.lower()
)


map_dict = {
    "quince": "15.0",
    "veinte": "20.0",
    "diez": "10.0"
}
df["Supervisor District"] = df["Supervisor District"].replace(map_dict)


df["Supervisor District"] = df["Supervisor District"].apply(lambda x: float(x) if pd.notna(x) else x)

## Hasta n < 15 valores distintos


## Address

Se identificaron múltiples columnas que aportaban información fragmentada de la dirección:

- **`Street Name`**  
- **`Street Number`**  
- **`Street Number Suffix`**  
- **`Street Suffix`**  
- **`Lot`**  
- **`Block`**  
- **`Zipcode`**  
- **`Unit`**  
- **`Unit Suffix`**  
- **`Neighborhoods - Analysis Boundaries`**

Con el fin de evitar redundancias y simplificar el análisis, se decidió **unificar toda esta información en una única columna final** llamada **`Address`**.

In [8]:

cols_src = [
    "Street Name", "Street Number", "Street Number Suffix", "Street Suffix",
    "Lot", "Block", "Zipcode",
    "Unit", "Unit Suffix", "Neighborhoods - Analysis Boundaries"
]
for c in cols_src:
    if c not in df.columns:
        df[c] = pd.NA


def _to_upper_str(s):
    return s.astype("string").str.strip().str.upper()

df["Street Name"] = _to_upper_str(df["Street Name"])
df["Street Suffix"] = _to_upper_str(df["Street Suffix"])
df["Street Number Suffix"] = _to_upper_str(df["Street Number Suffix"])
df["Unit"] = _to_upper_str(df["Unit"])
df["Unit Suffix"] = _to_upper_str(df["Unit Suffix"])
df["Neighborhoods - Analysis Boundaries"] = _to_upper_str(df["Neighborhoods - Analysis Boundaries"])


def _norm_token(x):
    if pd.isna(x):
        return pd.NA
    s = str(x).strip()
    if s.endswith(".0"):
        s = s[:-2]
    return s if s != "" else pd.NA

for col in ["Street Number", "Lot", "Block", "Unit", "Unit Suffix"]:
    df[col] = df[col].apply(_norm_token).astype("string")


def _norm_zip(x):
    if pd.isna(x):
        return pd.NA
    s = str(x).strip()
    if s.endswith(".0"):
        s = s[:-2]
    digits = re.sub(r"\D", "", s)
    if digits == "":
        return pd.NA
    if len(digits) >= 9:
        return f"{digits[:5]}-{digits[5:9]}"
    elif len(digits) >= 5:
        return digits[:5]
    else:
        return digits  

df["Zipcode"] = df["Zipcode"].apply(_norm_zip).astype("string")


def _compose_street(r):
    parts = [r["Street Number"], r["Street Number Suffix"], r["Street Name"], r["Street Suffix"]]
    parts = [p for p in parts if pd.notna(p) and p != ""]
    return " ".join(parts) if parts else pd.NA

street_part = (
    df.apply(_compose_street, axis=1)
      .astype("string")
      .str.replace(r"\s+", " ", regex=True)
      .str.strip()
)

def _compose_unit(u, usfx):
    parts = [p for p in [u, usfx] if pd.notna(p) and p != ""]
    if parts:
        token = "".join(parts)  
        return f"UNIT {token}"
    return pd.NA

unit_full = [
    _compose_unit(u, us)
    for u, us in zip(df["Unit"], df["Unit Suffix"])
]
unit_full = pd.Series(unit_full, index=df.index, dtype="string")


street_plus_unit = street_part.where(unit_full.isna() | (unit_full == ""), street_part + " " + unit_full)

block_part = df["Block"].apply(lambda v: f"BLOCK {v}" if pd.notna(v) and v != "" else pd.NA).astype("string")
lot_part   = df["Lot"].apply(lambda v: f"LOT {v}"   if pd.notna(v) and v != "" else pd.NA).astype("string")
neigh_part = df["Neighborhoods - Analysis Boundaries"].astype("string").apply(
    lambda v: v if pd.notna(v) and v != "" else pd.NA
)
zip_part   = df["Zipcode"].apply(lambda v: v if pd.notna(v) and v != "" else pd.NA).astype("string")


def _join_with_commas(*vals):
    vals = [v for v in vals if pd.notna(v) and str(v) != ""]
    return ", ".join(vals) if vals else pd.NA

df["Address"] = [
    _join_with_commas(su, b, l, n, z)
    for su, b, l, n, z in zip(street_plus_unit, block_part, lot_part, neigh_part, zip_part)
]


df["Address"] = (
    df["Address"]
      .astype("string")
      .str.replace(r"\s+", " ", regex=True)
      .str.replace(r"\s+,", ",", regex=True)
      .str.replace(r",\s*,", ", ", regex=True)
      .str.strip()
)

df = df.drop(columns=cols_src)

## Existing Use y Proposed Use

In [9]:
combos = (
    df[["Existing Use", "Proposed Use"]]
    .value_counts(dropna=False)
    .reset_index(name="count")
)

num_unique = combos.shape[0]
combos

,Existing Use,Proposed Use,count
0,1 family dwelling,1 family dwelling,45381
1,apartments,apartments,40447
2,NaN,NaN,38806
3,office,office,23403
4,2 family dwelling,2 family dwelling,20133
...,...,...,...
641,vacant lot,animal sale or care,1
642,lending institution,clinics-medic/dental,1
643,lending institution,church,1
644,lending institution,apartments,1


## Existing Units y Proposed Units

In [10]:
# Existing Units: NaN → 0
if "Existing Units" in df.columns:
    df["Existing Units"] = df["Existing Units"].fillna(0).astype(int)

# Proposed Units: 0 → NaN
if "Proposed Units" in df.columns:
    df["Proposed Units"] = df["Proposed Units"].replace(0, np.nan)

## Estimated Cost and Revised Cost

In [11]:
df["Esimated Cost"]
df["Revised Cost"]

for col in ["Estimated Cost", "Revised Cost"]:
    if col in df.columns:
        df[col] = df[col].replace([0, 1], np.nan)

KeyError: 'Esimated Cost'

## Plansets

In [ ]:
df["Plansets"]

if "Plansets" in df.columns:
    df["Plansets"] = df["Plansets"].fillna(0).astype(int)

## Number of Existing Stories y Number of Proposed Stories

In [ ]:
df["Number of Existing Stories"]
df["Number of Proposed Stories"]

# Existing Stories: NaN -> 0
if "Number of Existing Stories" in df.columns:
    df["Number of Existing Stories"] = (
        df["Number of Existing Stories"].fillna(0).astype(int)
    )

# Proposed Stories: 0 -> NaN
if "Number of Proposed Stories" in df.columns:
    df["Number of Proposed Stories"] = (
        df["Number of Proposed Stories"].replace(0, np.nan).astype("Int64")
    )

## Record ID

In [ ]:
df = df.drop(columns="Record ID")

## Site Permit y Fire only Permit

In [ ]:
def _permit_combo(row):
    site = row["Site Permit"] == "Y"
    fire = row["Fire Only Permit"] == "Y"
    if site and fire:
        return "SITE & FIRE"
    elif site:
        return "SITE PERMIT"
    elif fire:
        return "FIRE ONLY PERMIT"
    else:
        return pd.NA

# Crear la columna unificada
df["Special Permit"] = df.apply(_permit_combo, axis=1)

# Eliminar las columnas originales
df = df.drop(columns=["Site Permit", "Fire Only Permit"])

In [13]:
df.to_csv("nuevo.csv", index=False)